# 05 — Interactive Displacement Risk Map
**Project:** Charlotte NPA Housing Affordability Analysis  
**Purpose:** Translates model predictions into a browser-based interactive map for city planners and non-technical stakeholders.  

This notebook produces a single HTML file (`displacement_risk_map.html`) that any city planner can open in a browser without installing Python or any software. Each NPA polygon is colored by displacement risk. Clicking any NPA shows a popup with the key data points a planner needs to make resource allocation decisions.

In [1]:
import subprocess
subprocess.run(['pip', 'install', 'pyogrio', '--upgrade', '--break-system-packages'], capture_output=True)

CompletedProcess(args=['pip', 'install', 'pyogrio', '--upgrade', '--break-system-packages'], returncode=0, stdout=b'Requirement already satisfied: pyogrio in c:\\users\\juant\\anaconda3\\lib\\site-packages (0.12.1)\r\nRequirement already satisfied: certifi in c:\\users\\juant\\anaconda3\\lib\\site-packages (from pyogrio) (2025.8.3)\r\nRequirement already satisfied: numpy in c:\\users\\juant\\anaconda3\\lib\\site-packages (from pyogrio) (2.1.3)\r\nRequirement already satisfied: packaging in c:\\users\\juant\\appdata\\roaming\\python\\python313\\site-packages (from pyogrio) (25.0)\r\n', stderr=b'')

In [2]:
# Install required libraries if not already installed
import subprocess
subprocess.run(['pip', 'install', 'folium', 'geopandas', '--quiet'], capture_output=True)

import os
import sqlite3
import pandas as pd
import geopandas as gpd
import folium
from folium.features import GeoJsonTooltip, GeoJsonPopup

base = os.path.abspath(os.path.join(os.getcwd(), '..'))
db_path = os.path.join(base, 'data', 'charlotte_housing.db')
geojson_path = os.path.join(base, 'data', 'npa_boundaries.geojson')
output_path = os.path.join(base, 'outputs', 'displacement_risk_map.html')

os.makedirs(os.path.join(base, 'outputs'), exist_ok=True)

print(f'Base:     {base}')
print(f'Database: {db_path}')
print(f'GeoJSON:  {geojson_path}')
print(f'Output:   {output_path}')

Base:     c:\Users\juant\OneDrive\Desktop\Charlotte-NPA-Housing-Affordability-Analysis
Database: c:\Users\juant\OneDrive\Desktop\Charlotte-NPA-Housing-Affordability-Analysis\data\charlotte_housing.db
GeoJSON:  c:\Users\juant\OneDrive\Desktop\Charlotte-NPA-Housing-Affordability-Analysis\data\npa_boundaries.geojson
Output:   c:\Users\juant\OneDrive\Desktop\Charlotte-NPA-Housing-Affordability-Analysis\outputs\displacement_risk_map.html


In [3]:
# Load model predictions and NPA features from database
conn = sqlite3.connect(db_path)

# Load classification predictions saved in notebook 04
preds = pd.read_sql('SELECT * FROM classification_test_predictions', conn)

# Load full feature table for additional context in popups
features = pd.read_sql('SELECT npa_id, home_sales_price, household_income, \
                        absenteeism_pct, bachelors_pct, median_rent \
                        FROM npa_features_model', conn)

# Load the full displacement risk label for all NPAs (not just test set)
all_labels = pd.read_sql('SELECT npa_id, displacement_risk FROM npa_features_model', conn)

conn.close()

print(f'Predictions loaded: {len(preds)} test NPAs')
print(f'Features loaded: {len(features)} NPAs')
print(f'All labels loaded: {len(all_labels)} NPAs')
print(f'\nPrediction columns: {list(preds.columns)}')

Predictions loaded: 89 test NPAs
Features loaded: 444 NPAs
All labels loaded: 444 NPAs

Prediction columns: ['npa_id', 'actual_class', 'predicted_class', 'predicted_proba_atrisk']


In [4]:
# Merge predictions with full feature context
# For test NPAs we have model predictions, for train NPAs we use the actual label

# Start with all NPAs and their true labels
map_data = all_labels.merge(features, on='npa_id', how='left')

# Add predicted probability for test NPAs where available
preds_slim = preds[['npa_id', 'predicted_proba_atrisk', 'predicted_class']].copy()
preds_slim.columns = ['npa_id', 'predicted_proba_atrisk', 'predicted_class']

map_data = map_data.merge(preds_slim, on='npa_id', how='left')

# For NPAs not in test set, use the true label as the display risk
map_data['display_risk'] = map_data['displacement_risk']

# Create readable labels
map_data['risk_label'] = map_data['display_risk'].map({1: 'At Risk', 0: 'Stable'})

# Format dollar columns for display
map_data['home_price_fmt'] = map_data['home_sales_price'].apply(
    lambda x: f'${x:,.0f}' if pd.notna(x) else 'N/A'
)
map_data['income_fmt'] = map_data['household_income'].apply(
    lambda x: f'${x:,.0f}' if pd.notna(x) else 'N/A'
)
map_data['rent_fmt'] = map_data['median_rent'].apply(
    lambda x: f'${x:,.0f}/mo' if pd.notna(x) else 'N/A'
)
map_data['proba_fmt'] = map_data['predicted_proba_atrisk'].apply(
    lambda x: f'{x:.1%}' if pd.notna(x) else 'N/A (train NPA)'
)

print(f'Map data shape: {map_data.shape}')
print(f'At Risk NPAs: {(map_data["display_risk"] == 1).sum()}')
print(f'Stable NPAs: {(map_data["display_risk"] == 0).sum()}')
print(f'NPAs with null risk: {map_data["display_risk"].isna().sum()}')

Map data shape: (444, 15)
At Risk NPAs: 173
Stable NPAs: 271
NPAs with null risk: 0


In [5]:
# Load and merge NPA boundary GeoJSON
if not os.path.exists(geojson_path):
    raise FileNotFoundError(f"GeoJSON file not found at {geojson_path}. Please ensure npa_boundaries.geojson exists in the data folder.")

gdf = gpd.read_file(geojson_path)

# Ensure NPA_ID is integer for joining
gdf['NPA_ID'] = gdf['NPA_ID'].astype(int)
map_data['npa_id'] = map_data['npa_id'].astype(int)

# Merge model data onto geometry
gdf_merged = gdf.merge(map_data, left_on='NPA_ID', right_on='npa_id', how='left')

print(f'GeoJSON NPAs: {len(gdf)}')
print(f'Merged shape: {gdf_merged.shape}')
print(f'NPAs with risk label: {gdf_merged["risk_label"].notna().sum()}')
print(f'NPAs missing risk label: {gdf_merged["risk_label"].isna().sum()}')

GeoJSON NPAs: 459
Merged shape: (459, 20)
NPAs with risk label: 444
NPAs missing risk label: 15


In [6]:
# Fill any remaining nulls for display
gdf_merged['risk_label'] = gdf_merged['risk_label'].fillna('No Data')
gdf_merged['home_price_fmt'] = gdf_merged['home_price_fmt'].fillna('N/A')
gdf_merged['income_fmt'] = gdf_merged['income_fmt'].fillna('N/A')
gdf_merged['rent_fmt'] = gdf_merged['rent_fmt'].fillna('N/A')
gdf_merged['proba_fmt'] = gdf_merged['proba_fmt'].fillna('N/A')
gdf_merged['absenteeism_pct'] = gdf_merged['absenteeism_pct'].fillna('N/A')
gdf_merged['bachelors_pct'] = gdf_merged['bachelors_pct'].fillna('N/A')

# Color function
def get_color(risk_label):
    if risk_label == 'At Risk':
        return '#C0392B'   # firebrick red
    elif risk_label == 'Stable':
        return '#2980B9'   # steel blue
    else:
        return '#BDC3C7'   # light gray for no data

print('Colors assigned:')
print('  At Risk   = Red  (#C0392B)')
print('  Stable    = Blue (#2980B9)')
print('  No Data   = Gray (#BDC3C7)')

Colors assigned:
  At Risk   = Red  (#C0392B)
  Stable    = Blue (#2980B9)
  No Data   = Gray (#BDC3C7)


In [7]:
# Build the interactive Folium map
# Centered on Charlotte, NC
m = folium.Map(
    location=[35.2271, -80.8431],
    zoom_start=11,
    tiles='CartoDB positron',
    control_scale=True
)

# Add NPA polygons with color and popup
for _, row in gdf_merged.iterrows():
    color = get_color(row['risk_label'])

    # Build popup HTML — this is what city planners see when they click an NPA
    popup_html = f"""
    <div style="font-family: Arial, sans-serif; width: 260px; font-size: 13px;">
        <div style="background-color: {color}; color: white; padding: 8px 12px;
                    border-radius: 4px 4px 0 0; font-weight: bold; font-size: 15px;">
            NPA {int(row['NPA_ID'])} &nbsp;|&nbsp; {row['risk_label']}
        </div>
        <div style="padding: 10px 12px; background: #f9f9f9; border: 1px solid #ddd;
                    border-top: none; border-radius: 0 0 4px 4px;">
            <table style="width: 100%; border-collapse: collapse;">
                <tr>
                    <td style="padding: 4px 0; color: #555;">City</td>
                    <td style="padding: 4px 0; font-weight: bold; text-align: right;">{row['City']}</td>
                </tr>
                <tr style="background: #f0f0f0;">
                    <td style="padding: 4px 0; color: #555;">Avg Home Sales Price</td>
                    <td style="padding: 4px 0; font-weight: bold; text-align: right;">{row['home_price_fmt']}</td>
                </tr>
                <tr>
                    <td style="padding: 4px 0; color: #555;">Median Household Income</td>
                    <td style="padding: 4px 0; font-weight: bold; text-align: right;">{row['income_fmt']}</td>
                </tr>
                <tr style="background: #f0f0f0;">
                    <td style="padding: 4px 0; color: #555;">Median Rent</td>
                    <td style="padding: 4px 0; font-weight: bold; text-align: right;">{row['rent_fmt']}</td>
                </tr>
                <tr>
                    <td style="padding: 4px 0; color: #555;">Student Absenteeism</td>
                    <td style="padding: 4px 0; font-weight: bold; text-align: right;">{row['absenteeism_pct']}%</td>
                </tr>
                <tr style="background: #f0f0f0;">
                    <td style="padding: 4px 0; color: #555;">Bachelor's Degree %</td>
                    <td style="padding: 4px 0; font-weight: bold; text-align: right;">{row['bachelors_pct']}%</td>
                </tr>
                <tr>
                    <td style="padding: 4px 0; color: #555;">Model Risk Probability</td>
                    <td style="padding: 4px 0; font-weight: bold; text-align: right;">{row['proba_fmt']}</td>
                </tr>
            </table>
        </div>
    </div>
    """

    folium.GeoJson(
        row['geometry'].__geo_interface__,
        style_function=lambda x, c=color: {
            'fillColor': c,
            'color': 'white',
            'weight': 0.8,
            'fillOpacity': 0.70,
        },
        highlight_function=lambda x: {
            'weight': 2.5,
            'color': '#333',
            'fillOpacity': 0.90,
        },
        tooltip=folium.Tooltip(f"NPA {int(row['NPA_ID'])} — {row['risk_label']}"),
        popup=folium.Popup(folium.Html(popup_html, script=True), max_width=280),
    ).add_to(m)

print('All NPA polygons added to map.')

All NPA polygons added to map.


In [8]:
# Add a legend
legend_html = """
<div style="
    position: fixed;
    bottom: 40px; left: 40px;
    z-index: 1000;
    background-color: white;
    border: 2px solid #ccc;
    border-radius: 8px;
    padding: 14px 18px;
    font-family: Arial, sans-serif;
    font-size: 13px;
    box-shadow: 2px 2px 8px rgba(0,0,0,0.2);
">
    <b style="font-size: 14px;">Displacement Risk</b><br>
    <i style="background: #C0392B; width: 16px; height: 16px;
              display: inline-block; border-radius: 3px; margin-right: 8px;
              vertical-align: middle;"></i> At Risk<br><br>
    <i style="background: #2980B9; width: 16px; height: 16px;
              display: inline-block; border-radius: 3px; margin-right: 8px;
              vertical-align: middle;"></i> Stable<br><br>
    <i style="background: #BDC3C7; width: 16px; height: 16px;
              display: inline-block; border-radius: 3px; margin-right: 8px;
              vertical-align: middle;"></i> No Data<br>
    <hr style="margin: 10px 0; border: none; border-top: 1px solid #ddd;">
    <span style="color: #777; font-size: 11px;">
        Click any NPA for details.<br>
        Data: Mecklenburg QoL Explorer 2023.<br>
        Model: DTSC 2302 Charlotte NPA Project.
    </span>
</div>
"""

m.get_root().html.add_child(folium.Element(legend_html))

# Add a title banner
title_html = """
<div style="
    position: fixed;
    top: 10px; left: 50%; transform: translateX(-50%);
    z-index: 1000;
    background-color: #2C3E50;
    color: white;
    padding: 10px 24px;
    border-radius: 6px;
    font-family: Arial, sans-serif;
    font-size: 15px;
    font-weight: bold;
    box-shadow: 2px 2px 8px rgba(0,0,0,0.3);
    text-align: center;
">
    Charlotte NPA Displacement Risk Map &nbsp;|&nbsp;
    <span style="font-weight: normal; font-size: 13px;">
        Based on 2023 Mecklenburg County QoL Data
    </span>
</div>
"""

m.get_root().html.add_child(folium.Element(title_html))

print('Legend and title added.')

Legend and title added.


In [9]:
# Save the map as an HTML file
m.save(output_path)
print(f'Map saved to: {output_path}')
print(f'File size: {os.path.getsize(output_path) / 1024:.1f} KB')
print(f'\nShare this file with city planners.')
print(f'They can open it in any browser with no software installation required.')

Map saved to: c:\Users\juant\OneDrive\Desktop\Charlotte-NPA-Housing-Affordability-Analysis\outputs\displacement_risk_map.html
File size: 6440.8 KB

Share this file with city planners.
They can open it in any browser with no software installation required.


In [10]:
# Preview the map inline in the notebook
from IPython.display import IFrame
IFrame(output_path, width='100%', height=600)